# Phase 1: Data Foundation — Step 1.3: Data Cleaning

This notebook cleans the raw datasets by addressing the validation failures found in Step 1.2 (specifically the invalid employee ages in `hr_performance_engagement.csv`) and formats, splits, and saves the cleaned datasets to `data/processed/` for future phases.

In [1]:
import os
import pandas as pd

raw_dir = os.path.join("data", "raw")
processed_dir = os.path.join("data", "processed")
os.makedirs(processed_dir, exist_ok=True)
print(f"Raw directory: {os.path.abspath(raw_dir)}")
print(f"Processed directory: {os.path.abspath(processed_dir)}")

Raw directory: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai\data\raw
Processed directory: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai\data\processed


## 1. Cleaning Employee Attrition (`employee_attrition.csv` -> `employees.csv`)
We load the raw data, verify there are no issues, and save it directly to `data/processed/employees.csv`.

In [2]:
df_attr = pd.read_csv(os.path.join(raw_dir, "employee_attrition.csv"))
print(f"Initial shape: {df_attr.shape}")
df_attr.to_csv(os.path.join(processed_dir, "employees.csv"), index=False)
print("Saved employees.csv to processed/")

Initial shape: (1470, 35)
Saved employees.csv to processed/


## 2. Cleaning HR Performance & Engagement (`hr_performance_engagement.csv`)
We drop the 2 rows with Age = 17 (Employee IDs 1743 and 2038) to satisfy the constraint Age >= 18. We then split the dataset into `engagement_data.csv` and `performance_history.csv`.

In [3]:
df_perf = pd.read_csv(os.path.join(raw_dir, "hr_performance_engagement.csv"))
print(f"Initial shape: {df_perf.shape}")

# Drop rows where Age is less than 18
invalid_ages = df_perf[(df_perf["Age"] < 18) | (df_perf["Age"] > 100)]
print(f"Dropping {len(invalid_ages)} rows with Age < 18: {invalid_ages[['Employee ID', 'Age']].to_dict(orient='records')}")
df_perf_clean = df_perf.drop(invalid_ages.index)
print(f"Cleaned shape: {df_perf_clean.shape}")

# Save engagement survey columns
engagement_cols = ["Employee ID", "Survey Date", "Engagement Score", "Satisfaction Score", "Work-Life Balance Score"]
df_engagement = df_perf_clean[engagement_cols]
df_engagement.to_csv(os.path.join(processed_dir, "engagement_data.csv"), index=False)
print(f"Saved engagement_data.csv with shape: {df_engagement.shape}")

# Save performance and training columns
perf_history_cols = [
    "Employee ID", "Performance Score", "Current Employee Rating",
    "Training Date", "Training Program Name", "Training Type",
    "Training Outcome", "Training Duration(Days)", "Training Cost"
]
df_perf_history = df_perf_clean[perf_history_cols]
df_perf_history.to_csv(os.path.join(processed_dir, "performance_history.csv"), index=False)
print(f"Saved performance_history.csv with shape: {df_perf_history.shape}")

Initial shape: (2845, 28)
Dropping 2 rows with Age < 18: [{'Employee ID': 1743, 'Age': 17}, {'Employee ID': 2038, 'Age': 17}]
Cleaned shape: (2843, 28)
Saved engagement_data.csv with shape: (2843, 5)
Saved performance_history.csv with shape: (2843, 9)


## 3. Cleaning Occupation Data (`occupation_data.csv` -> `occupation_master.csv`)
We load the taxonomy, make no changes (as none were needed), and save it to `data/processed/occupation_master.csv`.

In [4]:
df_occ = pd.read_csv(os.path.join(raw_dir, "occupation_data.csv"))
print(f"Initial shape: {df_occ.shape}")
df_occ.to_csv(os.path.join(processed_dir, "occupation_master.csv"), index=False)
print("Saved occupation_master.csv to processed/")

Initial shape: (1016, 3)
Saved occupation_master.csv to processed/


## 4. Cleaning Essential Skills (`essential_skills.csv` -> `essential_skills_processed.csv`)
We fill the missing values in the `Not Relevant` column (which has 9,100 nulls) with `'N'` representing relevant.

In [5]:
df_ess = pd.read_csv(os.path.join(raw_dir, "essential_skills.csv"))
print(f"Initial shape: {df_ess.shape}")
print(f"Nulls in Not Relevant before: {df_ess['Not Relevant'].isnull().sum()}")
df_ess["Not Relevant"] = df_ess["Not Relevant"].fillna("N")
print(f"Nulls in Not Relevant after fillna: {df_ess['Not Relevant'].isnull().sum()}")
df_ess.to_csv(os.path.join(processed_dir, "essential_skills_processed.csv"), index=False)
print("Saved essential_skills_processed.csv to processed/")

Initial shape: (18200, 15)
Nulls in Not Relevant before: 9100
Nulls in Not Relevant after fillna: 0
Saved essential_skills_processed.csv to processed/


## 5. Cleaning Software Skills (`software_skills.csv` -> `software_skills_processed.csv`)
We save the software skills taxonomy directly without modifications.

In [6]:
df_soft = pd.read_csv(os.path.join(raw_dir, "software_skills.csv"))
print(f"Initial shape: {df_soft.shape}")
df_soft.to_csv(os.path.join(processed_dir, "software_skills_processed.csv"), index=False)
print("Saved software_skills_processed.csv to processed/")

Initial shape: (31821, 7)
Saved software_skills_processed.csv to processed/


## 6. Verification of Processed Files
We verify that all 6 processed files exist in `data/processed/` and confirm their final shapes.

In [7]:
processed_files = [
    "employees.csv",
    "engagement_data.csv",
    "performance_history.csv",
    "occupation_master.csv",
    "essential_skills_processed.csv",
    "software_skills_processed.csv"
]
for f in processed_files:
    path = os.path.join(processed_dir, f)
    df = pd.read_csv(path)
    print(f"Verified {f}: shape={df.shape}")

Verified employees.csv: shape=(1470, 35)
Verified engagement_data.csv: shape=(2843, 5)
Verified performance_history.csv: shape=(2843, 9)
Verified occupation_master.csv: shape=(1016, 3)
Verified essential_skills_processed.csv: shape=(18200, 15)
Verified software_skills_processed.csv: shape=(31821, 7)
